# Example 12.4 / 1: Price One Column from the Master Duals

Companion notebook for **12.4 Column Generation** in the AMPL lab Documentation, section "Price One Column from the Master Duals". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Price One Column from the Master Duals

The master below is the cutting-stock master of Cutting Stock Example (AMPL lab documentation, section 12.4.1): rolls of
width 10, pieces of width 3, 4 and 5 with demand 5, 4 and 3, and three seed patterns that each cut
one width as often as it fits. The run performs one iteration: it solves the master LP, reads the
row duals with `get_values("dual")` and the reduced costs with `get_values("rc")` (both defined on
Duals and Reduced Costs (AMPL lab documentation, section 3.6)), checks the reduced-cost formula on every master column,
prices all 11 patterns that fit a roll, and appends the best one by assigning `nPatterns` and the
new `pieces` entries:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                     # item types cut from stock rolls
param width {ITEMS} > 0;                       # item width
param stock_width > 0;                         # width of one stock roll
param demand {ITEMS} integer >= 0;             # pieces ordered per item
param nPatterns integer >= 0;                  # columns generated so far
set PATTERNS = 1..nPatterns;                   # current pattern pool
param pieces {ITEMS, PATTERNS} integer >= 0;   # pieces of item i in pattern p

var UsePattern {PATTERNS} >= 0;                # continuous during generation

minimize Rolls: sum {p in PATTERNS} UsePattern[p];  # restricted master: rolls used
subject to MeetDemand {i in ITEMS}:            # duals price items for the pricing step
    sum {p in PATTERNS} pieces[i,p] * UsePattern[p] >= demand[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["width"] = {"small": 3, "medium": 4, "large": 5}
ampl.param["stock_width"] = 10
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}
ampl.param["nPatterns"] = 3
ampl.param["pieces"] = {
    ("small", 1): 3, ("medium", 1): 0, ("large", 1): 0,
    ("small", 2): 0, ("medium", 2): 2, ("large", 2): 0,
    ("small", 3): 0, ("medium", 3): 0, ("large", 3): 2,
}

In [5]:
# run.py
from itertools import product

ampl.option["presolve"] = 0     # AMPL presolve can alter deduced duals; see the next box
items = ampl.set["ITEMS"].to_list()
width = ampl.param["width"].to_dict()
roll = ampl.param["stock_width"].value()
candidates = [dict(zip(items, c)) for c in product(range(4), repeat=len(items))
              if any(c) and sum(width[i] * k for i, k in zip(items, c)) <= roll]

ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved", ampl.solve_result
price = ampl.con["MeetDemand"].get_values("dual").to_dict()
rc = ampl.var["UsePattern"].get_values("rc").to_dict()
pieces = ampl.param["pieces"].to_dict()
for p in [1, 2, 3]:              # reconcile the formula with the solver suffix
    computed = 1 - sum(price[i] * pieces[i, p] for i in items)
    assert abs(computed - rc[p]) < 1e-9, (p, computed, rc[p])
best = max(candidates, key=lambda c: sum(price[i] * c[i] for i in items))
reduced = 1 - sum(price[i] * best[i] for i in items)
print(f"rolls {ampl.obj['Rolls'].value():.4g}, duals {price}")
print(f"best of {len(candidates)} patterns {best}, reduced cost {reduced:.4g}")

ampl.param["nPatterns"] = 4
ampl.param["pieces"] = {(i, 4): best[i] for i in items}
ampl.solve(solver="highs", verbose=False)
print(f"after adding it: rolls {ampl.obj['Rolls'].value():.4g}")

rolls 5.167, duals {'large': 0.5, 'medium': 0.5, 'small': 0.3333333333333333}
best of 11 patterns {'small': 2, 'medium': 1, 'large': 0}, reduced cost -0.1667
after adding it: rolls 4.75


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The script displays the reconciliation for the
seed columns, prices the pattern (2, 1, 0) found above, appends it, and solves again:

In [6]:
%%writefile data.dat
set ITEMS := small medium large;
param width := small 3  medium 4  large 5;
param stock_width := 10;
param demand := small 5  medium 4  large 3;
param nPatterns := 3;
param pieces:  1  2  3 :=
    small      3  0  0
    medium     0  2  0
    large      0  0  2 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0, presolve 0;
param candidate {ITEMS} integer >= 0;  # pieces per item in the priced pattern
solve;
display MeetDemand.dual;
display {p in PATTERNS}
    (UsePattern[p].rc, 1 - sum {i in ITEMS} MeetDemand[i].dual * pieces[i,p]);
let candidate['small'] := 2;
let candidate['medium'] := 1;
let candidate['large'] := 0;
display 1 - sum {i in ITEMS} MeetDemand[i].dual * candidate[i];
let nPatterns := nPatterns + 1;
let {i in ITEMS} pieces[i, nPatterns] := candidate[i];
solve;
display Rolls, MeetDemand.dual;

MeetDemand.dual [*] :=
 large  0.5
medium  0.5
 small  0.333333
;

: UsePattern[p].rc 1 - sum{i in ITEMS} (MeetDemand[i].dual)*pieces[i,p]    :=
1         0                                  0
2         0                                  0
3         0                                  0
;

1 - sum{i in ITEMS} (MeetDemand[i].dual)*candidate[i] = -0.166667

Rolls = 4.75

MeetDemand.dual [*] :=
 large  0.5
medium  0.5
 small  0.25
;



### Another way in amplpy

The first master costs 5.167 rolls with duals 1/3, 1/2 and 1/2. The pattern with two small
pieces and one medium piece has value 2/3 + 1/2 = 7/6 and reduced cost -1/6, so it enters, and
the master drops to 4.75 rolls. One more pricing step finds no pattern with negative reduced
cost, so 4.75 is the optimum of the master LP over all 11 patterns; Cutting Stock
Example (AMPL lab documentation, section 12.4.1) runs that loop to the end with a knapsack pricing problem and then solves
the integer master.

**Pitfall: presolve changes the master duals.**

The reconciliation guards against more than sign errors. With AMPL's default presolve, the same
seed master returns duals that pass the formula check but fail a second one, that every column
used at a positive value has reduced cost 0:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["width"] = {"small": 3, "medium": 4, "large": 5}
ampl.param["stock_width"] = 10
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}
ampl.param["nPatterns"] = 3
ampl.param["pieces"] = {
    ("small", 1): 3, ("medium", 1): 0, ("large", 1): 0,
    ("small", 2): 0, ("medium", 2): 2, ("large", 2): 0,
    ("small", 3): 0, ("medium", 3): 0, ("large", 3): 2,
}

In [11]:
# run.py
ampl.param["nPatterns"] = 3         # back to the seed master
ampl.option["presolve"] = 10        # the AMPL default
ampl.solve(solver="highs", verbose=False)
price = ampl.con["MeetDemand"].get_values("dual").to_dict()
rc = ampl.var["UsePattern"].get_values("rc").to_dict()
use = ampl.var["UsePattern"].to_dict()
print("duals", price)
for p in [1, 2, 3]:
    computed = 1 - sum(price[i] * ampl.param["pieces"][i, p] for i in price)
    print(f"pattern {p}: value {use[p]:.4g}, computed reduced cost {computed:.4g}, rc {rc[p]:.4g}")

duals {'large': 0.5, 'medium': 0.5, 'small': 0}
pattern 1: value 1.667, computed reduced cost 1, rc 1
pattern 2: value 2, computed reduced cost 0, rc 0
pattern 3: value 1.5, computed reduced cost 0, rc 0
